# Silver: Sector Assignment

Assigns jobs to industry sectors using deterministic rules and fallback logic.

## Assignment Strategy

1. **Company-based** (deterministic): Match company to known sector mapping
2. **Title-based** (pattern matching): Use job title keywords
3. **Skill-based** (aggregation): Infer from extracted skills
4. **ML-based** (future): Predict sector from description
5. **Default**: Queue for manual review if confidence < threshold

## Sectors

- Technology, Finance, Healthcare, Retail, Manufacturing, Education, etc.
- Extendable taxonomy in `metadata.sector_dim`

## Architecture

**Input**: `silver.silver_jobs_current`  
**Output**: Updates `silver_jobs_current` with sector assignments  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.sector_assignment_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Updates jobs in current table with sector fields

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")
dbutils.widgets.text("confidence_threshold", "0.7", "Minimum confidence for auto-assignment")
dbutils.widgets.dropdown("queue_low_confidence", "true", ["true", "false"], "Queue low confidence for review")

batch_id = dbutils.widgets.get("batch_id").strip()
confidence_threshold = float(dbutils.widgets.get("confidence_threshold"))
queue_low_confidence = dbutils.widgets.get("queue_low_confidence") == "true"

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

# Sector assignment rules (company name patterns)
COMPANY_SECTOR_PATTERNS = {
    "technology": ["software", "tech", "data", "cloud", "ai", "digital", "cyber", "saas"],
    "finance": ["bank", "financial", "fintech", "insurance", "investment", "capital"],
    "healthcare": ["health", "medical", "pharma", "biotech", "hospital", "clinic"],
    "retail": ["retail", "ecommerce", "shop", "store", "marketplace"],
    "manufacturing": ["manufacturing", "automotive", "industrial", "engineering"],
    "education": ["education", "university", "school", "learning", "training"],
    "consulting": ["consulting", "advisory", "strategy"],
    "media": ["media", "publishing", "entertainment", "streaming"],
    "energy": ["energy", "oil", "gas", "renewable", "utility"],
    "transportation": ["logistics", "transport", "shipping", "delivery"]
}

# Title-based sector keywords
TITLE_SECTOR_KEYWORDS = {
    "technology": ["developer", "engineer", "data scientist", "devops", "software", "programmer", "architect"],
    "finance": ["accountant", "financial analyst", "auditor", "controller", "trader"],
    "healthcare": ["nurse", "doctor", "clinician", "therapist", "medical"],
    "education": ["teacher", "professor", "instructor", "tutor"],
    "consulting": ["consultant", "advisor", "analyst"],
    "marketing": ["marketing", "brand manager", "seo", "content"],
    "sales": ["sales", "account executive", "business development"]
}

print(f"Loaded {len(COMPANY_SECTOR_PATTERNS)} sector categories")

In [0]:
# Create metadata table to track sector-assigned batches
metadata_table = f"{METADATA_SCHEMA}.sector_assignment_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  jobs_processed INT,
  high_confidence_count INT,
  low_confidence_count INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been sector-assigned'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("jobs_processed", IntegerType(), True),
    StructField("high_confidence_count", IntegerType(), True),
    StructField("low_confidence_count", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in current table
    all_batches_query = f"""
        SELECT DISTINCT current_batch_id as batch_id, source_name
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE is_active = true AND soft_delete_flag = false
        AND current_batch_id IS NOT NULL
        AND current_batch_id != ''
        AND source_name IS NOT NULL
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in current but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "total_assigned": 0})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
def assign_sector_by_company(company_name):
    """Assign sector based on company name patterns"""
    if not company_name:
        return None, 0.0
    
    company_lower = company_name.lower()
    
    for sector, patterns in COMPANY_SECTOR_PATTERNS.items():
        for pattern in patterns:
            if pattern in company_lower:
                return sector, 0.85  # High confidence for company match
    
    return None, 0.0

def assign_sector_by_title(title):
    """Assign sector based on job title keywords"""
    if not title:
        return None, 0.0
    
    title_lower = title.lower()
    
    for sector, keywords in TITLE_SECTOR_KEYWORDS.items():
        for keyword in keywords:
            if keyword in title_lower:
                return sector, 0.75  # Moderate confidence for title match
    
    return None, 0.0

def assign_sector_combined(company, title):
    """Combined sector assignment with fallback"""
    # Try company first (higher confidence)
    sector_company, conf_company = assign_sector_by_company(company)
    if sector_company and conf_company >= 0.70:
        return (sector_company, "company_pattern", conf_company)
    
    # Fallback to title
    sector_title, conf_title = assign_sector_by_title(title)
    if sector_title:
        return (sector_title, "title_keyword", conf_title)
    
    # Default: unknown
    return ("unknown", "no_match", 0.0)

# Register UDF
sector_assign_udf = F.udf(assign_sector_combined, StructType([
    StructField("sector", StringType()),
    StructField("assignment_method", StringType()),
    StructField("confidence", DoubleType())
]))

print("Sector assignment functions registered")

In [0]:
# Initialize tracking
total_jobs_processed = 0
total_high_confidence = 0
total_low_confidence = 0
processed_batch_count = 0
failed_batches = []

# Process each batch
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Assigning sectors for batch {current_batch_id} ({current_source})...", end=" ")
        
        # Load jobs for this batch
        batch_query = f"""
            SELECT 
                enterprise_job_id,
                source_name,
                company_name_norm,
                title_normalized,
                description_raw
            FROM {SILVER_SCHEMA}.silver_jobs_current
            WHERE current_batch_id = '{current_batch_id}'
            AND is_active = true AND soft_delete_flag = false
        """
        
        if current_source:
            batch_query += f" AND source_name = '{current_source}'"
        
        jobs_df = spark.sql(batch_query)
        job_count = jobs_df.count()
        
        if job_count == 0:
            print("No jobs found")
            continue
        
        # Assign sectors
        sector_assigned_df = jobs_df.withColumn(
            "sector_assignment",
            sector_assign_udf(F.col("company_name_norm"), F.col("title_normalized"))
        ).select(
            "*",
            F.col("sector_assignment.sector").alias("assigned_sector"),
            F.col("sector_assignment.assignment_method").alias("assignment_method"),
            F.col("sector_assignment.confidence").alias("assignment_confidence")
        ).drop("sector_assignment")
        
        # Add metadata
        sector_assigned_df = sector_assigned_df.withColumn(
            "assigned_at", run_timestamp
        ).withColumn(
            "sector_batch_id", F.lit(current_batch_id)
        )
        
        # Count confidence levels
        high_conf = sector_assigned_df.filter(F.col("assignment_confidence") >= confidence_threshold).count()
        low_conf = sector_assigned_df.filter(F.col("assignment_confidence") < confidence_threshold).count()
        
        # Update current table with sector assignments
        from delta.tables import DeltaTable
        
        # Ensure sector columns exist in target table
        try:
            current_schema = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").schema
            current_columns = [field.name for field in current_schema.fields]
            
            missing_columns = []
            if "sector_assigned" not in current_columns:
                missing_columns.append("ADD COLUMN sector_assigned STRING")
            if "sector_assignment_method" not in current_columns:
                missing_columns.append("ADD COLUMN sector_assignment_method STRING")
            if "sector_confidence" not in current_columns:
                missing_columns.append("ADD COLUMN sector_confidence DECIMAL(5,4)")
            if "sector_assigned_at" not in current_columns:
                missing_columns.append("ADD COLUMN sector_assigned_at TIMESTAMP")
            
            if missing_columns:
                for col_stmt in missing_columns:
                    spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_jobs_current {col_stmt}")
        except:
            pass
        
        # Merge sector assignments into current table
        delta_table = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
        
        delta_table.alias("target").merge(
            sector_assigned_df.alias("source"),
            "target.enterprise_job_id = source.enterprise_job_id"
        ).whenMatchedUpdate(set={
            "sector_assigned": "source.assigned_sector",
            "sector_assignment_method": "source.assignment_method",
            "sector_confidence": F.expr("CAST(source.assignment_confidence AS DECIMAL(5,4))"),
            "sector_assigned_at": "source.assigned_at"
        }).execute()
        
        # Record batch processing in metadata
        safe_source = current_source if current_source else "unknown"
        metadata_record = spark.createDataFrame([
            (current_batch_id, safe_source, int(job_count), int(high_conf), int(low_conf), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_jobs_processed += job_count
        total_high_confidence += high_conf
        total_low_confidence += low_conf
        processed_batch_count += 1
        
        print(f"✓ Jobs:{job_count} High:{high_conf} Low:{low_conf}")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            safe_source = current_source if current_source else "unknown"
            failure_record = spark.createDataFrame([
                (current_batch_id, safe_source, 0, 0, 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nProcessed {processed_batch_count} batches, {total_jobs_processed} jobs assigned sectors")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

In [0]:
# Show sector assignment history
if processed_batch_count > 0:
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, jobs_processed, high_confidence_count, low_confidence_count, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show current sector distribution
    sector_summary_df = spark.sql(f"""
        SELECT 
            sector_assigned,
            COUNT(*) as job_count,
            ROUND(AVG(sector_confidence), 3) as avg_confidence
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE sector_assigned IS NOT NULL
        GROUP BY sector_assigned
        ORDER BY job_count DESC
    """)
    display(sector_summary_df)
    
    # Show assignment methods
    method_summary_df = spark.sql(f"""
        SELECT 
            sector_assignment_method,
            COUNT(*) as count
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE sector_assignment_method IS NOT NULL
        GROUP BY sector_assignment_method
        ORDER BY count DESC
    """)
    display(method_summary_df)

# Queue low confidence jobs for review if enabled
if queue_low_confidence and total_low_confidence > 0:
    low_confidence_jobs = spark.sql(f"""
        SELECT enterprise_job_id, source_name, sector_assigned, sector_confidence
        FROM {SILVER_SCHEMA}.silver_jobs_current
        WHERE sector_confidence < {confidence_threshold}
        AND sector_assigned IS NOT NULL
    """)
    
    # Create review queue table if needed
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_semantic_review_queue (
      review_id STRING,
      enterprise_job_id STRING,
      issue_type STRING,
      issue_detail STRING,
      confidence DECIMAL(5,4),
      queued_at TIMESTAMP,
      status STRING
    )
    USING DELTA
    """)
    
    # Add missing columns if needed (schema evolution)
    try:
        current_schema = spark.table(f"{SILVER_SCHEMA}.silver_semantic_review_queue").schema
        current_columns = [field.name for field in current_schema.fields]
        
        missing_columns = []
        if "resolved_at" not in current_columns:
            missing_columns.append("ADD COLUMN resolved_at TIMESTAMP")
        if "resolution_notes" not in current_columns:
            missing_columns.append("ADD COLUMN resolution_notes STRING")
        
        if missing_columns:
            for col_stmt in missing_columns:
                spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_semantic_review_queue {col_stmt}")
    except:
        pass
    
    # Queue for review
    review_records = low_confidence_jobs.select(
        F.expr("uuid()").alias("review_id"),
        "enterprise_job_id",
        F.lit("SECTOR_LOW_CONFIDENCE").alias("issue_type"),
        F.concat(
            F.lit("Sector '"),
            F.col("sector_assigned"),
            F.lit("' assigned with confidence "),
            F.col("sector_confidence").cast("string")
        ).alias("issue_detail"),
        F.col("sector_confidence").cast("decimal(5,4)").alias("confidence"),
        run_timestamp.alias("queued_at"),
        F.lit("PENDING").alias("status"),
        F.lit(None).cast("timestamp").alias("resolved_at"),
        F.lit(None).cast("string").alias("resolution_notes")
    )
    
    review_records.write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_semantic_review_queue")
    print(f"Queued {total_low_confidence} low-confidence jobs for review")

# Exit summary
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "jobs_processed": total_jobs_processed,
    "high_confidence_count": total_high_confidence,
    "low_confidence_count": total_low_confidence,
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))